In [ ]:
# -*- coding: utf-8 -*-

# ==========================================
# 1. INSTALL & IMPORT LIBRARIES
# ==========================================
!pip install openpyxl scikit-optimize shap xgboost joblib -q

import os
import gc
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from google.colab import drive

from sklearn.model_selection import (
    StratifiedGroupKFold,
    cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, auc,
    precision_score, recall_score, f1_score
)
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 2.0 MB/s eta 0:00:00


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

mpl.rcParams.update({
    "font.family":         "DejaVu Sans",
    "font.size":           11,
    "axes.titlesize":      12,
    "axes.titleweight":    "bold",
    "axes.labelsize":      11,
    "axes.linewidth":      1.0,
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "xtick.labelsize":     10,
    "ytick.labelsize":     10,
    "legend.fontsize":     10,
    "legend.frameon":      True,
    "legend.framealpha":   0.9,
    "legend.edgecolor":    "0.5",
    "figure.dpi":          120,
    "savefig.dpi":         600,
    "savefig.bbox":        "tight",
    "savefig.pad_inches":  0.05,
    "pdf.fonttype":        42,
    "ps.fonttype":         42,
})

MODEL_COLOURS = {
    "RF":       "#2E86AB",
    "SVM":      "#E07A5F",
    "XGB":      "#7E4E90",
    "Ensemble": "#0E1B2C",
}

PRETTY_FEATURE = {
    "aspect":         "Aspect",
    "curvature":      "Curvature (profile)",
    "dis_to_fault":   "Distance to fault",
    "dis_to_river":   "Distance to river",
    "dis_to_road":    "Distance to road",
    "dis_to_thrust":  "Distance to thrust",
    "ndvi":           "NDVI",
    "slope":          "Slope",
    "twi":            "TWI",
    "aspect_average":        "Aspect (mean)",
    "aspect_stddev":         "Aspect (sd)",
    "curvature_average":     "Curvature (mean)",
    "curvature_stddev":      "Curvature (sd)",
    "dis_to_fault_average":  "Dist. to fault (mean)",
    "dis_to_fault_stddev":   "Dist. to fault (sd)",
    "dis_to_river_average":  "Dist. to river (mean)",
    "dis_to_river_stddev":   "Dist. to river (sd)",
    "dis_to_road_average":   "Dist. to road (mean)",
    "dis_to_road_stddev":    "Dist. to road (sd)",
    "dis_to_thrust_average": "Dist. to thrust (mean)",
    "dis_to_thrust_stddev":  "Dist. to thrust (sd)",
    "ndvi_average":          "NDVI (mean)",
    "ndvi_stddev":           "NDVI (sd)",
    "slope_average":         "Slope (mean)",
    "slope_stddev":          "Slope (sd)",
    "twi_average":           "TWI (mean)",
    "twi_stddev":            "TWI (sd)",
}

def pretty(name):
    return PRETTY_FEATURE.get(name, name)


In [ ]:


# Mount Drive; silently force-remount if the FUSE endpoint dropped.
try:
    drive.mount('/content/drive')
except Exception:
    drive.mount('/content/drive', force_remount=True)

# ==========================================
# 2. PATHS
# ==========================================
excel_path = "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/parameters/Filtered_Excel files/"
excel_files = {
    "3x3": "ls_non_ls_final_3x3",
    "15x15": "ls_non_ls_final_15x15",
    "25x25": "ls_non_ls_final_25x25"
}

base_output_dir = "/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/model_outputs_spatial"
os.makedirs(base_output_dir, exist_ok=True)

# ==========================================
# 3. SETTINGS
# ==========================================
RANDOM_STATE = 42
TEST_SIZE = 0.2

# spatial CV
INNER_CV_SPLITS = 5
OUTER_CV_SPLITS = 5

BAYES_N_ITER = 20

# spatial block size in same unit as X,Y (meters)
SPATIAL_BLOCK_SIZE = 1000

# SHAP sampling controls
SHAP_BACKGROUND_SIZE = 100
SHAP_EXPLAIN_SIZE = 100

# ==========================================
# 4. HELPERS
# ==========================================
def make_dir(path):
    os.makedirs(path, exist_ok=True)

def sanitize_columns(df):
    df.columns = df.columns.str.strip()
    return df

def plot_correlation_matrix(df, save_path, title):
    corr = df.rename(columns={c: pretty(c) for c in df.columns}).corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(8.5, 7))
    sns.heatmap(
        corr, ax=ax,
        cmap="RdBu_r", center=0, vmin=-1, vmax=1,
        annot=True, fmt=".2f", annot_kws={"fontsize": 8},
        square=True, linewidths=0.4, linecolor="white",
        cbar_kws={"shrink": 0.75, "label": "Pearson r"},
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    ax.set_title(title)
    fig.tight_layout()
    safe_savefig(fig, save_path)
    plt.show()
    plt.close(fig)

def plot_distributions_by_class(df, features, target_col, out_dir, grid_name):
    """Single 3xN composite KDE figure (one panel per feature)."""
    df_plot = df.copy()
    df_plot["class"] = df_plot[target_col].map({0: "Non-landslide", 1: "Landslide"})

    feats = list(features)
    n = len(feats)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.2 * nrows))
    axes = np.atleast_2d(axes).flatten()

    for ax, col in zip(axes, feats):
        sns.kdeplot(
            data=df_plot, x=col, hue="class",
            fill=True, common_norm=False, alpha=0.45, linewidth=1.6,
            palette={"Landslide": "#C0392B", "Non-landslide": "#2E86AB"},
            ax=ax, legend=False,
        )
        ax.set_title(pretty(col))
        ax.set_xlabel("")
        ax.set_ylabel("Density")
        ax.grid(alpha=0.25, linestyle=":")

    for ax in axes[n:]:
        ax.axis("off")

    handles = [
        mpatches.Patch(facecolor="#C0392B", alpha=0.5, label="Landslide"),
        mpatches.Patch(facecolor="#2E86AB", alpha=0.5, label="Non-landslide"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(f"Conditioning-factor distributions by class - {grid_name}", y=1.00)
    fig.tight_layout(rect=(0, 0.03, 1, 0.97))

    save_path = os.path.join(out_dir, f"{grid_name}_distributions.png")
    safe_savefig(fig, save_path)
    plt.show()
    plt.close(fig)

def save_confusion_matrix(cm, labels, title, save_path):
    cm = np.asarray(cm)
    cm_pct = cm.astype(float) / cm.sum() * 100.0
    annot = np.array([[f"{cm[i, j]:d}\n({cm_pct[i, j]:.1f}%)"
                       for j in range(cm.shape[1])]
                      for i in range(cm.shape[0])])
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    sns.heatmap(
        cm, ax=ax, annot=annot, fmt="", cmap="Blues",
        xticklabels=labels, yticklabels=labels,
        cbar=False, annot_kws={"fontsize": 10},
        linewidths=0.4, linecolor="white",
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    fig.tight_layout()
    safe_savefig(fig, save_path)
    plt.show()
    plt.close(fig)

def save_roc_curve(y_true, y_prob, title, save_path, color="#2E86AB"):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(5.0, 4.2))
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f"AUC = {auc_score:.3f}")
    ax.plot([0, 1], [0, 1], color="0.4", linestyle="--", linewidth=1)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(title)
    ax.grid(alpha=0.25, linestyle=":")
    ax.legend(loc="lower right")
    fig.tight_layout()
    safe_savefig(fig, save_path)
    plt.show()
    plt.close(fig)


def save_combined_roc(probs_dict, y_test, grid_name, save_path):
    """Overlay ROC curves for all models on one panel.

    probs_dict: {"RF": np.array, "SVM": np.array, "XGB": np.array, "Ensemble": np.array}
    """
    fig, ax = plt.subplots(figsize=(6.0, 5.0))
    ax.plot([0, 1], [0, 1], color="0.4", linestyle="--", linewidth=1)
    for name, prob in probs_dict.items():
        fpr, tpr, _ = roc_curve(y_test, prob)
        ax.plot(fpr, tpr, linewidth=1.8,
                color=MODEL_COLOURS.get(name, None),
                label=f"{name} (AUC = {auc(fpr, tpr):.3f})")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"ROC curves - {grid_name}")
    ax.grid(alpha=0.25, linestyle=":")
    ax.legend(loc="lower right")
    fig.tight_layout()
    safe_savefig(fig, save_path)
    plt.show()
    plt.close(fig)

def compute_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auc": float(roc_auc_score(y_true, y_prob))
    }

def create_spatial_groups(coords_df, block_size=1000):
    """
    Create spatial block IDs from X,Y coordinates.
    Nearby samples fall into same block, so train/test won't mix nearby locations.
    """
    block_x = np.floor(coords_df["X"].values / block_size).astype(int)
    block_y = np.floor(coords_df["Y"].values / block_size).astype(int)
    groups = pd.Series(block_x.astype(str) + "_" + block_y.astype(str), index=coords_df.index)
    return groups

def compute_spatial_cv_auc_stats(estimator, X, y, groups, n_splits=5, random_state=42):
    cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    auc_scores = cross_val_score(
        estimator,
        X,
        y,
        cv=cv,
        groups=groups,
        scoring="roc_auc",
        n_jobs=-1
    )
    return float(np.mean(auc_scores)), float(np.std(auc_scores)), auc_scores

def plot_calibration_comparison(model, calibrated_model, X_test, y_test, grid_name, model_name, save_dir):
    """Two-panel reliability diagram: raw vs Platt-calibrated probabilities."""
    from sklearn.calibration import calibration_curve as _cal_curve

    raw_prob = model.predict_proba(X_test)[:, 1]
    cal_prob = calibrated_model.predict_proba(X_test)[:, 1]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    for ax, prob, label in zip(axes, [raw_prob, cal_prob], ["Raw", "Calibrated (Platt)"]):
        frac_pos, mean_pred = _cal_curve(y_test, prob, n_bins=8, strategy="quantile")
        ax.plot([0, 1], [0, 1], color="0.4", linestyle="--", linewidth=1)
        ax.plot(mean_pred, frac_pos, "o-", color="#2E86AB", linewidth=1.8, markersize=6)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_xlabel("Mean predicted probability")
        ax.set_ylabel("Observed fraction of landslides")
        ax.set_title(label)
        ax.grid(alpha=0.25, linestyle=":")
    fig.suptitle(f"{model_name} calibration - {grid_name}", y=1.00)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    safe_savefig(fig, os.path.join(save_dir, f"{model_name}_{grid_name}_calibration.png"))
    plt.show()
    plt.close(fig)

def plot_feature_importance(importances, feature_names, title, save_path, top_n=15, color="#2E86AB"):
    imp_df = (pd.DataFrame({"feature": [pretty(f) for f in feature_names],
                            "importance": importances})
                .sort_values("importance", ascending=False).head(top_n))
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(imp_df) + 1.5))
    sns.barplot(data=imp_df, x="importance", y="feature",
                color=color, edgecolor="white", linewidth=0.5, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Mean decrease in Gini / gain (normalised)")
    ax.set_ylabel("Conditioning factor")
    ax.grid(axis="x", alpha=0.25, linestyle=":")
    fig.tight_layout()
    safe_savefig(fig, save_path)
    plt.show()
    plt.close(fig)

    return imp_df

def get_native_importance(model_name, fitted_estimator, X_data, y_data):
    if model_name == "RF":
        return fitted_estimator.feature_importances_
    elif model_name == "XGB":
        return fitted_estimator.feature_importances_
    elif model_name == "SVM":
        perm = permutation_importance(
            fitted_estimator,
            X_data,
            y_data,
            scoring="f1",
            n_repeats=10,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        return perm.importances_mean
    else:
        raise ValueError("Unknown model name")

def ensemble_predict_proba(model_dict, X):
    probs = []
    for _, model in model_dict.items():
        probs.append(model.predict_proba(X)[:, 1])
    return np.mean(np.column_stack(probs), axis=1)

def plot_shap_tree(model, X_sample, title, save_prefix):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)

    # Normalise SHAP output to a 2-D (n_samples, n_features) array for the
    # positive (landslide) class. shap/sklearn versions differ in what
    # TreeExplainer returns:
    #   * list [class0, class1] of 2-D arrays           (older shap)
    #   * 3-D ndarray (n_samples, n_features, n_classes) (newer shap on RF)
    #   * plain 2-D ndarray                              (single-output, e.g. XGB)
    # Without this, RandomForest's 3-D output reached summary_plot and was
    # drawn as a clipped interaction-style matrix instead of a beeswarm.
    shap_plot_values = shap_values
    if isinstance(shap_plot_values, list):
        shap_plot_values = (shap_plot_values[1]
                            if len(shap_plot_values) > 1 else shap_plot_values[0])
    shap_plot_values = np.asarray(shap_plot_values)
    if shap_plot_values.ndim == 3:
        cls = 1 if shap_plot_values.shape[-1] > 1 else 0
        shap_plot_values = shap_plot_values[..., cls]

    plt.figure()
    shap.summary_plot(shap_plot_values, X_sample, show=False)
    plt.title(title)
    plt.tight_layout()
    safe_savefig(plt, f"{save_prefix}_shap_summary.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

def plot_shap_kernel(model, X_background, X_sample, title, save_prefix):
    predict_fn = lambda data: model.predict_proba(
        pd.DataFrame(data, columns=X_sample.columns)
    )[:, 1]

    X_bg = shap.sample(
        X_background,
        min(len(X_background), SHAP_BACKGROUND_SIZE),
        random_state=RANDOM_STATE
    )
    X_used = shap.sample(
        X_sample,
        min(len(X_sample), SHAP_EXPLAIN_SIZE),
        random_state=RANDOM_STATE
    )

    explainer = shap.KernelExplainer(predict_fn, X_bg)
    shap_values = explainer.shap_values(X_used, nsamples=100)

    plt.figure()
    shap.summary_plot(shap_values, X_used, show=False)
    plt.title(title)
    plt.tight_layout()
    safe_savefig(plt, f"{save_prefix}_shap_summary.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

def plot_ensemble_permutation_importance(models_dict, X_test, y_test, save_path, title):
    baseline_prob = ensemble_predict_proba(models_dict, X_test)
    baseline_pred = (baseline_prob >= 0.5).astype(int)
    baseline_f1 = f1_score(y_test, baseline_pred, zero_division=0)

    importances = []
    rng = np.random.default_rng(RANDOM_STATE)

    for col in X_test.columns:
        scores = []
        for _ in range(10):
            X_perm = X_test.copy()
            X_perm[col] = rng.permutation(X_perm[col].values)
            prob = ensemble_predict_proba(models_dict, X_perm)
            pred = (prob >= 0.5).astype(int)
            score = f1_score(y_test, pred, zero_division=0)
            scores.append(baseline_f1 - score)
        importances.append(np.mean(scores))

    return plot_feature_importance(
        np.array(importances),
        list(X_test.columns),
        title=title,
        save_path=save_path
    )

def save_prediction_table(grid_out_dir, grid, y_test, y_pred, y_prob):
    results_df = pd.DataFrame({
        "Actual": y_test,
        "Predicted": y_pred,
        "Probability": y_prob
    }).reset_index(drop=True)
    results_df.to_csv(os.path.join(grid_out_dir, f"{grid}_prediction.csv"), index=False)
    print(results_df.head(20))
    return results_df

def plot_ensemble_shap(models_dict, X_train, X_test, save_prefix, title):
    X_bg = shap.sample(X_train, min(len(X_train), SHAP_BACKGROUND_SIZE), random_state=RANDOM_STATE)
    X_explain = shap.sample(X_test, min(len(X_test), SHAP_EXPLAIN_SIZE), random_state=RANDOM_STATE)

    predict_fn = lambda data: ensemble_predict_proba(
        models_dict,
        pd.DataFrame(data, columns=X_test.columns)
    )

    explainer = shap.KernelExplainer(predict_fn, X_bg)
    shap_values = explainer.shap_values(X_explain, nsamples=100)

    plt.figure()
    shap.summary_plot(shap_values, X_explain, show=False)
    plt.title(title)
    plt.tight_layout()
    safe_savefig(plt, f"{save_prefix}_shap_summary.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

def save_json(data, path):
    with open(path, "w") as f:
        json.dump(data, f, indent=4)


def safe_savefig(fig_or_plt, path, **kwargs):
    """Save a figure, auto-remounting Drive on Errno 107 (ENOTCONN).

    Catches the 'Transport endpoint is not connected' OSError that arises
    when the Google Drive FUSE mount drops during a long Colab session,
    force-remounts Drive, then retries the save once.
    """
    def _do():
        fig_or_plt.savefig(path, **kwargs)

    try:
        _do()
    except OSError as e:
        if e.errno == 107 or "Transport endpoint" in str(e):
            import os as _os
            print(f"⚠️  Drive disconnected while saving "
                  f"{_os.path.basename(path)} — remounting…")
            from google.colab import drive as _drive
            _drive.mount('/content/drive', force_remount=True)
            print("✅  Drive remounted — retrying save…")
            _do()
        else:
            raise


def autodrop_correlated(X, threshold=0.8, label="", prefer_drop=("roughness", "ndwi")):
    """Iteratively drop features where |Pearson r| > threshold.
    Returns
    X_clean : DataFrame
    dropped_list : list of (dropped_col, retained_col, r_value) tuples
    """
    cols = list(X.columns)
    dropped = []

    while True:
        corr = X[cols].corr(numeric_only=True).abs()
        found = False
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                r = corr.iloc[i, j]
                if r > threshold:
                    di = any(tok in cols[i] for tok in prefer_drop)
                    dj = any(tok in cols[j] for tok in prefer_drop)
                    if di != dj:
                        to_drop, to_keep = (cols[i], cols[j]) if di else (cols[j], cols[i])
                    else:
                        ci = int((corr[cols[i]] > threshold).sum()) - 1
                        cj = int((corr[cols[j]] > threshold).sum()) - 1
                        if ci > cj or (ci == cj and cols[i] > cols[j]):
                            to_drop, to_keep = cols[i], cols[j]
                        else:
                            to_drop, to_keep = cols[j], cols[i]
                    dropped.append((to_drop, to_keep, round(float(r), 4)))
                    cols.remove(to_drop)
                    found = True
                    break
            if found:
                break
        if not found:
            break

    tag = f" [{label}]" if label else ""
    if dropped:
        print(f"[autodrop{tag}] Removed {len(dropped)} feature(s) with |r| > {threshold}:")
        for d, k, rv in dropped:
            print(f"  DROP '{d}'  (|r|={rv} with '{k}')")
    else:
        print(f"[autodrop{tag}] No features exceed |r| > {threshold}.")
    return X[cols].copy(), dropped

# ==========================================
# 5. MODELS + SEARCH SPACES
# ==========================================
models = {
    "RF": RandomForestClassifier(random_state=RANDOM_STATE),
    "SVM": SVC(probability=True, class_weight="balanced", random_state=RANDOM_STATE),
    "XGB": XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        use_label_encoder=False
    )
}

param_spaces = {
    "RF": {
        "n_estimators": Integer(150, 400),
        "max_depth": Integer(5, 25),
        "min_samples_split": Integer(2, 10),
        "min_samples_leaf": Integer(1, 10),
        "class_weight": Categorical(["balanced", "balanced_subsample"])
    },
    "SVM": {
        "model__C": Real(1e-2, 100, prior="log-uniform"),
        "model__gamma": Real(1e-5, 10, prior="log-uniform"),
        "model__kernel": Categorical(["rbf"])
    },
    "XGB": {
        "n_estimators": Integer(150, 400),
        "max_depth": Integer(3, 12),
        "learning_rate": Real(0.01, 0.3, prior="log-uniform"),
        "subsample": Real(0.6, 1.0),
        "colsample_bytree": Real(0.6, 1.0)
    }
}

# ==========================================
# 6. LOOP OVER ALL GRIDS
# ==========================================
all_grid_summary = []

for grid, file in excel_files.items():
    print(f"\n{'='*25} GRID: {grid} {'='*25}\n")

    # --------------------------------------
    # Paths for this grid
    # --------------------------------------
    grid_out_dir = os.path.join(base_output_dir, grid)
    make_dir(grid_out_dir)
    make_dir(os.path.join(grid_out_dir, "models"))
    make_dir(os.path.join(grid_out_dir, "plots"))
    make_dir(os.path.join(grid_out_dir, "plots", "roc"))
    make_dir(os.path.join(grid_out_dir, "plots", "cm"))
    make_dir(os.path.join(grid_out_dir, "plots", "importance"))
    make_dir(os.path.join(grid_out_dir, "plots", "shap"))

    # --------------------------------------
    # Load data
    # --------------------------------------
    df = pd.read_excel(f"{excel_path}{file}.xlsx")
    df = sanitize_columns(df)

    # keep X,Y for spatial grouping
    required_cols = ["ID", "X", "Y"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{file}.xlsx is missing required columns: {missing_cols}")

    # -- Gross correlation heatmap (all features, before auto-drop) --------
    df = df.dropna().reset_index(drop=True)
    X_for_corr = df.drop(columns=["ID", "X", "Y"], errors="ignore").copy()
    plot_correlation_matrix(
        X_for_corr,
        save_path=os.path.join(grid_out_dir, "plots", f"{grid}_gross_correlation_matrix.png"),
        title=f"All-features Correlation Matrix (pre-drop) - {grid}"
    )

    # -- Auto-drop features with |r| > 0.8 --------------------------------
    # ndwi and roughness are expected to be caught automatically.
    # Any additional correlated pairs above the threshold are also removed.
    _, dropped_features = autodrop_correlated(X_for_corr, threshold=0.8, label=grid)
    dropped_col_names = [d[0] for d in dropped_features]
    # Apply column drops then row-clean (same order as original)
    df = df.drop(columns=dropped_col_names, errors="ignore")
    df = df.dropna().reset_index(drop=True)

    coords = df[["X", "Y"]].copy()
    groups = create_spatial_groups(coords, block_size=SPATIAL_BLOCK_SIZE)

    # features exclude ID and coordinates
    y = df["ID"].astype(int).values
    X = df.drop(columns=["ID", "X", "Y"]).copy()

    print("Shape:", df.shape)
    print("Class balance:", np.bincount(y))
    print("Features (retained after auto-drop):", list(X.columns))
    print("Unique spatial groups:", groups.nunique())

    # -- Post-drop correlation heatmap (retained features only) ------------
    plot_correlation_matrix(
        X,
        save_path=os.path.join(grid_out_dir, "plots", f"{grid}_correlation_matrix.png"),
        title=f"Retained-features Correlation Matrix - {grid}"
    )

    plot_distributions_by_class(
        df=pd.concat([X, pd.Series(y, name="ID")], axis=1),
        features=X.columns,
        target_col="ID",
        out_dir=os.path.join(grid_out_dir, "plots"),
        grid_name=grid
    )

    # --------------------------------------
    # SPATIAL + STRATIFIED train-test split
    # --------------------------------------
    spatial_splitter = StratifiedGroupKFold(
        n_splits=int(round(1 / TEST_SIZE)),   # TEST_SIZE=0.2 -> 5 folds
        shuffle=True,
        random_state=RANDOM_STATE
    )

    train_idx, test_idx = next(spatial_splitter.split(X, y, groups=groups))

    X_train = X.iloc[train_idx].reset_index(drop=True)
    X_test = X.iloc[test_idx].reset_index(drop=True)
    y_train = y[train_idx]
    y_test = y[test_idx]

    groups_train = groups.iloc[train_idx].reset_index(drop=True)
    groups_test = groups.iloc[test_idx].reset_index(drop=True)

    print("Train size:", len(X_train), "| Test size:", len(X_test))
    print("Train groups:", groups_train.nunique(), "| Test groups:", groups_test.nunique())
    print("Train class balance:", np.bincount(y_train))
    print("Test class balance:", np.bincount(y_test))
    print("Train class ratio:", np.bincount(y_train) / len(y_train))
    print("Test class ratio:", np.bincount(y_test) / len(y_test))

    inner_cv = StratifiedGroupKFold(
        n_splits=INNER_CV_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    fitted_models = {}
    fitted_probs = {}      
    grid_results = []

    # ======================================
    # 7. TRAIN RF, SVM, XGB
    # ======================================
    for name, model in models.items():
        print(f"\n{'-'*15} Training {name} on {grid} {'-'*15}")

        if name == "SVM":
            estimator = Pipeline([
                ("scaler", StandardScaler()),
                ("model", model)
            ])
            search_space = param_spaces[name]
        else:
            estimator = model
            search_space = param_spaces[name]

        search = BayesSearchCV(
            estimator=estimator,
            search_spaces=search_space,
            n_iter=BAYES_N_ITER,
            cv=inner_cv,
            scoring="f1",
            n_jobs=-1,
            random_state=RANDOM_STATE,
            refit=True
        )

        search.fit(X_train, y_train, groups=groups_train)

        best_estimator = search.best_estimator_

        # Final calibrator with spatially-blocked Platt folds.
        _final_cal_cv = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE).split(X_train, y_train, groups=groups_train))
        calibrated = CalibratedClassifierCV(best_estimator, cv=_final_cal_cv)
        calibrated.fit(X_train, y_train)

        # holdout spatial test metrics
        y_pred = calibrated.predict(X_test)
        y_prob = calibrated.predict_proba(X_test)[:, 1]
        fitted_probs[name] = y_prob

        save_prediction_table(grid_out_dir, grid, y_test, y_pred, y_prob)

        metrics_dict = compute_metrics(y_test, y_pred, y_prob)
        cm = confusion_matrix(y_test, y_pred)

        # spatial CV AUC mean ± std on training data only
        best_estimator_for_cv = CalibratedClassifierCV(best_estimator, cv=3)

        mean_auc_cv, std_auc_cv, auc_scores_cv = compute_spatial_cv_auc_stats(
            best_estimator_for_cv,
            X_train,
            y_train,
            groups_train,
            n_splits=OUTER_CV_SPLITS,
            random_state=RANDOM_STATE
        )

        print(f"{name} Best Params: {search.best_params_}")
        print(f"{name} Holdout Metrics: {metrics_dict}")
        print(f"{name} Spatial CV AUC: {mean_auc_cv:.4f} ± {std_auc_cv:.4f}")
        print(f"{name} Fold-wise AUCs: {np.round(auc_scores_cv, 4)}")
        print("Confusion Matrix:\n", cm)
        print(classification_report(y_test, y_pred, digits=4))

        save_confusion_matrix(
            cm,
            labels=["Non-Landslide (0)", "Landslide (1)"],
            title=f"{name} Confusion Matrix - {grid}",
            save_path=os.path.join(grid_out_dir, "plots", "cm", f"{name}_{grid}_cm.png")
        )

        # keep ROC label based on test AUC only
        save_roc_curve(
            y_test,
            y_prob,
            title=f"{name} ROC Curve - {grid}",
            save_path=os.path.join(grid_out_dir, "plots", "roc", f"{name}_{grid}_roc.png")
        )

        # Calibration reliability diagram: raw vs Platt-calibrated
        plot_calibration_comparison(
            model=best_estimator,
            calibrated_model=calibrated,
            X_test=X_test,
            y_test=y_test,
            grid_name=grid,
            model_name=name,
            save_dir=os.path.join(grid_out_dir, "plots")
        )

        # feature importance
        fitted_base = best_estimator

        if name == "RF":
            importances = get_native_importance(name, fitted_base, X_train, y_train)
            plot_feature_importance(
                importances,
                X.columns,
                title=f"{name} Feature Importance - {grid}",
                save_path=os.path.join(grid_out_dir, "plots", "importance", f"{name}_{grid}_importance.png")
            )
            plot_shap_tree(
                fitted_base,
                shap.sample(X_test, min(len(X_test), SHAP_EXPLAIN_SIZE), random_state=RANDOM_STATE),
                title=f"{name} SHAP - {grid}",
                save_prefix=os.path.join(grid_out_dir, "plots", "shap", f"{name}_{grid}")
            )

        elif name == "XGB":
            importances = get_native_importance(name, fitted_base, X_train, y_train)
            plot_feature_importance(
                importances,
                X.columns,
                title=f"{name} Feature Importance - {grid}",
                save_path=os.path.join(grid_out_dir, "plots", "importance", f"{name}_{grid}_importance.png")
            )
            plot_shap_tree(
                fitted_base,
                shap.sample(X_test, min(len(X_test), SHAP_EXPLAIN_SIZE), random_state=RANDOM_STATE),
                title=f"{name} SHAP - {grid}",
                save_prefix=os.path.join(grid_out_dir, "plots", "shap", f"{name}_{grid}")
            )

        elif name == "SVM":
            # BUG FIX: was X_train/y_train — permutation importance must use held-out test set
            importances = get_native_importance(name, fitted_base, X_test, y_test)
            plot_feature_importance(
                importances,
                X.columns,
                title=f"{name} Permutation Importance - {grid}",
                save_path=os.path.join(grid_out_dir, "plots", "importance", f"{name}_{grid}_importance.png")
            )
            plot_shap_kernel(
                fitted_base,
                X_train,
                X_test,
                title=f"{name} SHAP - {grid}",
                save_prefix=os.path.join(grid_out_dir, "plots", "shap", f"{name}_{grid}")
            )

        model_path = os.path.join(grid_out_dir, "models", f"{name}_{grid}_calibrated.pkl")
        joblib.dump(calibrated, model_path)

        params_path = os.path.join(grid_out_dir, "models", f"{name}_{grid}_best_params.json")
        save_json(search.best_params_, params_path)

        fitted_models[name] = calibrated

        row = {
            "grid": grid,
            "model": name,
            **metrics_dict,
            "spatial_cv_auc_mean": mean_auc_cv,
            "spatial_cv_auc_std": std_auc_cv,
            "spatial_cv_auc_text": f"{mean_auc_cv:.4f} ± {std_auc_cv:.4f}",
            "model_path": model_path
        }
        grid_results.append(row)

        gc.collect()

    # ======================================
    # 8. ENSEMBLE
    # ======================================
    print(f"\n{'-'*15} Ensemble on {grid} {'-'*15}")

    ensemble_prob = ensemble_predict_proba(fitted_models, X_test)
    ensemble_pred = (ensemble_prob >= 0.5).astype(int)

    ensemble_metrics = compute_metrics(y_test, ensemble_pred, ensemble_prob)
    ensemble_cm = confusion_matrix(y_test, ensemble_pred)

    print("Ensemble Metrics:", ensemble_metrics)
    print("Ensemble Confusion Matrix:\n", ensemble_cm)
    print(classification_report(y_test, ensemble_pred, digits=4))

    save_confusion_matrix(
        ensemble_cm,
        labels=["Non-Landslide (0)", "Landslide (1)"],
        title=f"Ensemble Confusion Matrix - {grid}",
        save_path=os.path.join(grid_out_dir, "plots", "cm", f"Ensemble_{grid}_cm.png")
    )

    save_roc_curve(
        y_test,
        ensemble_prob,
        title=f"Ensemble ROC Curve - {grid}",
        save_path=os.path.join(grid_out_dir, "plots", "roc", f"Ensemble_{grid}_roc.png")
    )

    # Combined-ROC panel (all four models overlaid)
    fitted_probs["Ensemble"] = ensemble_prob
    save_combined_roc(
        fitted_probs, y_test, grid_name=grid,
        save_path=os.path.join(grid_out_dir, "plots", "roc", f"Combined_{grid}_roc.png"),
    )

    ensemble_imp_df = plot_ensemble_permutation_importance(
        fitted_models,
        X_test,
        y_test,
        save_path=os.path.join(grid_out_dir, "plots", "importance", f"Ensemble_{grid}_importance.png"),
        title=f"Ensemble Permutation Importance - {grid}"
    )

    plot_ensemble_shap(
        fitted_models,
        X_train,
        X_test,
        save_prefix=os.path.join(grid_out_dir, "plots", "shap", f"Ensemble_{grid}"),
        title=f"Ensemble SHAP - {grid}"
    )

    ensemble_bundle = {
        "models": fitted_models,
        "features": list(X.columns),
        "threshold": 0.5,
        "method": "mean_probability",
        "spatial_block_size": SPATIAL_BLOCK_SIZE
    }
    ensemble_path = os.path.join(grid_out_dir, "models", f"Ensemble_{grid}.pkl")
    joblib.dump(ensemble_bundle, ensemble_path)

    grid_results.append({
        "grid": grid,
        "model": "Ensemble",
        **ensemble_metrics,
        "model_path": ensemble_path
    })

    grid_summary_df = pd.DataFrame(grid_results)
    grid_summary_df.to_csv(os.path.join(grid_out_dir, f"{grid}_model_summary.csv"), index=False)

    all_grid_summary.extend(grid_results)

    print(f"\nCompleted grid: {grid}")
    print(grid_summary_df)

    gc.collect()

# ==========================================
# 9. FINAL SUMMARY ACROSS ALL GRIDS
# ==========================================
final_summary_df = pd.DataFrame(all_grid_summary)
final_summary_path = os.path.join(base_output_dir, "all_grids_model_summary.csv")
final_summary_df.to_csv(final_summary_path, index=False)

print("\nFinal summary across all grids:")
print(final_summary_df)
print(f"\nSaved final summary to: {final_summary_path}")


Output hidden; open in https://colab.research.google.com to view.

<!-- Audit fix L5 (2026-05-28): stale orphan code cell removed. The fragment that lived here duplicated logic already inside the main loop in the cell above; left as a no-op markdown cell so cell indices in the rest of the notebook do not shift. -->